# Verifying the label-path fixes (Blockers 1 & 2)

What this checks, on a **CPU runtime** — no GPU, no model weights, no Drive, ~1 minute:

| | Blocker | Fix |
|---|---|---|
| **1** | The teacher label depended on *which other examples* were in the dataset | `make_baseline_loo` — leave-one-out baselines |
| **2** | Sink correction happened in probability space, and `clamp_min(0)` flattened most of the label to a tie at zero | `pmi_scores` / `teacher_label` — log-space correction, dense, candidates excluded rather than zeroed |

The synthetic data below reproduces the structure the real WEAR-VQA run showed: **one dominant
attention sink that wins the argmax on every example**, plus a question-specific blob underneath it.
Because it is synthetic we know the true question signal, so we can *measure* which label recovers it —
not just eyeball heatmaps.

## 1. Setup

Self-contained: clones the repo from GitHub and nothing else. Nothing to upload.
It deletes any previous clone first, so re-running always picks up the latest `main`.

In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/shubhamOjha1000/text_vision_attention_map.git"
DEST = "/content/text_vision_attention_map"

shutil.rmtree(DEST, ignore_errors=True)              # always take a fresh copy
subprocess.run(["git", "clone", "-q", REPO, DEST], check=True)
os.chdir(DEST)
if DEST not in sys.path:
    sys.path.insert(0, DEST)

for m in ("visual_selection", "rater_selection"):    # drop stale imports on re-run
    sys.modules.pop(m, None)
import visual_selection as VS

NEW_API = ["make_baseline_loo", "pmi_scores", "candidate_mask", "teacher_label", "TeacherLabel"]
missing = [f for f in NEW_API if not hasattr(VS, f)]
assert not missing, f"clone is stale, missing: {missing}"

rev = subprocess.run(["git", "log", "-1", "--format=%h  %s"],
                     capture_output=True, text=True).stdout.strip()
print("repo @", rev)
print("new API available:", ", ".join(NEW_API))

## 2. Run the test suite

13 new tests cover the two fixes; the 8 pre-existing sink/selection tests must still pass
(the old `subtract_baseline` / `select_debiased` path is deliberately left working — it is
still the right thing for *selection*, just not for *labels*).

**These tests have never been executed** — there is no working Python on the machine where
the fix was written, so this cell is their first real run.

In [ ]:
!pip -q install pytest
!python -m pytest tests/test_visual_selection.py -q

## 3. A synthetic world that matches the real finding

Every example gets:

* a **shared, question-invariant position bias** with one dominant sink at patch 80
  (bottom-right on the 9x9 grid — exactly what the real run reported),
* a **question-specific Gaussian blob** — the ground truth a good label must recover,
* a little per-example noise.

In [ ]:
import torch

G, L_v, SINK = 9, 81, 80
torch.manual_seed(0)

shared = torch.randn(L_v) * 0.3
shared[SINK] += 4.0                      # the dominant, question-invariant sink

def blob(center, sigma=1.2):
    r, c = divmod(center, G)
    yy, xx = torch.meshgrid(torch.arange(G).float(), torch.arange(G).float(), indexing="ij")
    return torch.exp(-(((yy - r) ** 2 + (xx - c) ** 2) / (2 * sigma ** 2))).flatten()

def make_example(q_patch, seed):
    """-> (importance, true_signal). true_signal is the question-specific logit
    contribution we injected: what a faithful label should rank correctly."""
    g = torch.Generator().manual_seed(seed)
    signal = 2.5 * blob(q_patch)
    noise = torch.randn(L_v, generator=g) * 0.15
    return torch.softmax(shared + noise + signal, dim=0), signal

Q_PATCHES = [12, 30, 40, 22, 58, 4, 66, 35, 19, 47, 7, 71, 25, 52, 38, 60, 15, 44, 29, 55]
examples = [make_example(q, seed=i) for i, q in enumerate(Q_PATCHES)]
imps    = [e[0] for e in examples]
signals = [e[1] for e in examples]

peaks = [int(x.argmax()) for x in imps]
print(f"{len(imps)} examples | L_v = {L_v} ({G}x{G} grid)")
print("argmax of raw importance:", peaks[:10], "...")
print(f"all {peaks.count(SINK)}/{len(peaks)} examples peak at the sink (patch {SINK}).")
print("Raw importance is question-INVARIANT -> useless as a label. Same as the real run.")

## 4. Blocker 1 — the label must not correct itself

`make_baseline` averages every example, then subtracts that average from each one.
So example 0 supplies 1/N of its *own* correction, and its label shifts whenever the
dataset composition changes.

Test: edit example 0, then ask **whose baseline moved**.

In [ ]:
b_corpus = VS.make_baseline(imps)
b_loo    = VS.make_baseline_loo(imps)              # [N, L_v]; row i excludes example i

print(f"corpus baseline argmax : {int(b_corpus.argmax())}   (finds the sink)")
print(f"LOO row-0     argmax   : {int(b_loo[0].argmax())}   (finds it too)\n")

imps_edit = list(imps)
imps_edit[0] = make_example(3, seed=999)[0]        # change example 0 ONLY

d_corpus = (VS.make_baseline(imps)       - VS.make_baseline(imps_edit)).abs().max()
d_loo    = (VS.make_baseline_loo(imps)[0] - VS.make_baseline_loo(imps_edit)[0]).abs().max()

print("after editing example 0, example 0's own baseline moved by:")
print(f"   make_baseline      : {d_corpus:.6f}   <- self-contaminated")
print(f"   make_baseline_loo  : {d_loo:.6f}   <- independent of example 0")

**Caveat, stated honestly:** leave-one-out removes only the *self*-contribution. Row `i` still
depends on the other N-1 examples. It is the best you can do without extra model passes; a
**frozen held-out** baseline or a **per-image null-prompt** baseline is stronger, and the
null-prompt version is better still because it removes per-image position bias rather than only
the corpus-average sink.

## 5. Blocker 2 — which label actually recovers the question signal?

Because the ground-truth signal is known, this is measurable rather than a matter of taste.
Spearman correlation over all 81 patches, averaged across the 20 examples.

In [ ]:
from scipy.stats import spearmanr
import numpy as np

sub_r, pmi_r, zeros = [], [], []
for i in range(len(imps)):
    base, truth = b_loo[i], signals[i].numpy()
    sub = VS.subtract_baseline(imps[i], base).numpy()
    pmi = VS.pmi_scores(imps[i], base).numpy()
    sub_r.append(spearmanr(sub, truth)[0])
    pmi_r.append(spearmanr(pmi, truth)[0])
    zeros.append(int((sub == 0).sum()))
sub_r, pmi_r, zeros = np.array(sub_r), np.array(pmi_r), np.array(zeros)

print("Spearman vs the TRUE question signal (higher = better label):")
print(f"   subtract_baseline (old) : {sub_r.mean():.3f}")
print(f"   pmi_scores        (new) : {pmi_r.mean():.3f}")
print(f"\nentries clamped to exactly 0 by subtract_baseline: "
      f"{zeros.mean():.1f} / {L_v}   ({zeros.mean() / L_v:.0%} of the grid)")
print("   -> all of those are TIED at zero: the label cannot rank them at all,")
print("      so the student is told nothing about their relative importance.")

Why the log-space version wins is not a coincidence. The sink is an **additive bias on the
attention logits**, so

```
log p(i | q) - log p(i)  =  (shared + noise + signal) - (shared)  =  signal + noise + const
```

recovers the injected signal almost exactly. Subtracting in *probability* space is a nonlinear
operation on the wrong quantity, and the `clamp_min(0)` then discards the entire below-baseline
tail.

## 6. Candidates are excluded, not zeroed — and scores are stored raw

In [ ]:
lab = VS.teacher_label(imps[0], b_loo[0], drop_sink_k=3)
p = lab.distribution()

print(f"candidates            : {lab.n_cand} / {lab.L_v}")
print(f"sink {SINK} still a candidate : {bool(lab.cand_mask[SINK])}")
print(f"teacher distribution  : min={p.min():.2e}  max={p.max():.3f}  sum={p.sum():.4f}")
print("   -> dense: every candidate carries mass, so every candidate is ranked.\n")

# FRM spec: store RAW scores, normalise in the loss -> sweeps never regenerate labels
a = VS.teacher_label(imps[0], b_loo[0], drop_sink_k=0)
b = VS.teacher_label(imps[0], b_loo[0], drop_sink_k=8)
print(f"scores identical across drop_sink_k : {torch.equal(a.scores, b.scores)}")
print(f"   n_cand {a.n_cand} vs {b.n_cand} -> sweep sink-k / fovea radius, labels stay fixed\n")

p1, p2 = lab.distribution(0.5), lab.distribution(2.0)
print(f"temperature 0.5 max={p1.max():.3f} | 2.0 max={p2.max():.3f} | "
      f"ranking preserved: {torch.equal(p1.argsort(), p2.argsort())}")

**A correction to something I told you earlier.** I said a zeroed target lets the student dump
probability on the sink "at no cost". That was wrong: with a softmax student, the gradient of
`KL(p || r)` at a zero-target position is `r_i - 0 = r_i > 0`, which does push it down. The real
damage from clamping is the one measured in section 5 — most of the grid gets tied at zero, so the
label loses the ability to rank those patches. Excluding candidates is still the better design
(it drops the token from the student's keys too, and matches the FRM `cand` set), just not for the
reason I originally gave.

## 7. See it

In [ ]:
import matplotlib.pyplot as plt

i = 0
base = b_loo[i]
panels = [("TRUE question signal", signals[i]),
          ("raw importance", imps[i]),
          ("subtract_baseline (old)", VS.subtract_baseline(imps[i], base)),
          ("pmi_scores (new)", VS.pmi_scores(imps[i], base))]

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (title, v) in zip(axes, panels):
    im = ax.imshow(v.reshape(G, G), cmap="viridis")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

print(f"question patch = {Q_PATCHES[i]} (row {Q_PATCHES[i] // G}, col {Q_PATCHES[i] % G});  "
      f"sink = {SINK} (bottom-right)")

## What this notebook does NOT prove

It verifies the **plumbing**: labels are now per-example, dense, log-space corrected, and decoupled
from the candidate choice. On synthetic data where the answer is known by construction.

It says nothing about whether the *real* teacher labels are any good. Two things still open:

* **Blocker 3** — importance is still aggregated over **question** tokens. Per the FRM spec that is
  the control quantity (`imp_question`); the training label should come from **answer**-token rows
  (`imp_answer`), which is what reaches far context.
* **Blocker 4** — no faithfulness check. Mask patch `i`, measure the drop in `log P(answer)`,
  correlate against the label. Spearman ~0.5 means the label is a real attribution; ~0 means it is a
  saliency map with extra steps. Run this on ~50 examples **before** generating labels at scale.